# 25. Unsupervised Learning: Hierarchical Clustering

## Algorithm Category
**Type**: Unsupervised Learning - Clustering  
**Complexity**: Medium  
**Use Case**: Build hierarchy of clusters using agglomerative or divisive approach

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand hierarchical clustering and its variants (agglomerative, divisive)
- Implement agglomerative hierarchical clustering
- Understand linkage criteria (single, complete, average, ward)
- Visualize dendrograms
- Extract clusters at different levels
- Apply hierarchical clustering to real-world problems

## Historical Context

Hierarchical clustering has been used since the 1950s:
- Ward, J.H. (1963): "Hierarchical grouping to optimize an objective function"
- Johnson, S.C. (1967): "Hierarchical clustering schemes"
- One of the oldest clustering methods

**Key Papers/References:**
- Ward, J.H. (1963). "Hierarchical grouping to optimize an objective function"
- Johnson, S.C. (1967). "Hierarchical clustering schemes"

## When to Use Hierarchical Clustering

Hierarchical clustering is appropriate when:
- You want to explore cluster structure at multiple levels
- You don't know the number of clusters in advance
- You need to visualize cluster relationships
- Clusters may have nested structure
- You want interpretable cluster hierarchy
- Working with small to medium datasets

## Theory & Mechanics

### Mathematical Foundation

Hierarchical clustering builds a tree of clusters (dendrogram).

**Agglomerative (Bottom-Up):**
1. Start with each point as its own cluster
2. Merge closest clusters iteratively
3. Continue until all points in one cluster

**Divisive (Top-Down):**
1. Start with all points in one cluster
2. Split clusters iteratively
3. Continue until each point is its own cluster

**Linkage Criteria:**

1. **Single Linkage (Minimum):**
   $$d(C_i, C_j) = \min_{x \in C_i, y \in C_j} ||x - y||$$

2. **Complete Linkage (Maximum):**
   $$d(C_i, C_j) = \max_{x \in C_i, y \in C_j} ||x - y||$$

3. **Average Linkage:**
   $$d(C_i, C_j) = \frac{1}{|C_i||C_j|} \sum_{x \in C_i} \sum_{y \in C_j} ||x - y||$$

4. **Ward Linkage (Minimizes variance):**
   $$d(C_i, C_j) = \frac{|C_i||C_j|}{|C_i| + |C_j|} ||\mu_i - \mu_j||^2$$

### How It Works

**Agglomerative Algorithm:**
1. **Initialize**: Each point is a cluster
2. **Compute distances**: Calculate distance matrix between clusters
3. **Merge**: Merge two closest clusters
4. **Update**: Update distance matrix
5. **Repeat**: Steps 2-4 until one cluster remains

### Key Hyperparameters

- **n_clusters**: Number of clusters to extract (if specified)
- **linkage**: Linkage criterion ('ward', 'complete', 'average', 'single')
- **affinity**: Distance metric ('euclidean', 'manhattan', 'cosine', etc.)
- **distance_threshold**: Cut-off distance for flat clustering

### Advantages

- No need to specify number of clusters
- Produces interpretable dendrogram
- Works with any distance metric
- Can extract clusters at any level
- Handles non-spherical clusters better than K-Means

### Limitations

- Computationally expensive O(n³) for agglomerative
- Sensitive to noise and outliers
- Greedy algorithm (may not find global optimum)
- Difficult to scale to large datasets
- Memory intensive (stores full distance matrix)


## Implementation

Let's implement hierarchical clustering and visualize dendrograms.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score

# Import our helper functions
from src.models.unsupervised import hierarchical_cluster, evaluate_clustering

print("Libraries imported successfully!")


In [ ]:
# Generate synthetic dataset
X, y_true = make_blobs(n_samples=150, centers=4, n_features=2, 
                       random_state=42, cluster_std=0.60)

print(f"Dataset Shape: {X.shape}")
print(f"True number of clusters: {len(np.unique(y_true))}")

# Apply Agglomerative Clustering
clustering = AgglomerativeClustering(n_clusters=4, linkage='ward')
y_pred = clustering.fit_predict(X)

print(f"\nHierarchical Clustering Results:")
print(f"  Number of clusters: {clustering.n_clusters}")
print(f"  Linkage: {clustering.linkage}")

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title('True Clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
plt.title('Hierarchical Clustering (k=4, Ward)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Dendrogram Visualization

Let's create and analyze dendrograms for different linkage methods.


In [ ]:
# Create dendrograms for different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, method in enumerate(linkage_methods):
    row = idx // 2
    col = idx % 2
    
    # Compute linkage matrix
    linkage_matrix = linkage(X, method=method)
    
    # Plot dendrogram
    dendrogram(linkage_matrix, ax=axes[row, col], truncate_mode='level', p=5)
    axes[row, col].set_title(f'Dendrogram ({method.capitalize()} Linkage)')
    axes[row, col].set_xlabel('Sample Index')
    axes[row, col].set_ylabel('Distance')

plt.tight_layout()
plt.show()

# Extract clusters at different levels
print("\nClusters at different distance thresholds:")
for threshold in [2, 4, 6, 8]:
    clustering_thresh = AgglomerativeClustering(
        n_clusters=None, 
        distance_threshold=threshold,
        linkage='ward'
    )
    labels_thresh = clustering_thresh.fit_predict(X)
    n_clusters = len(np.unique(labels_thresh))
    print(f"  Distance threshold {threshold}: {n_clusters} clusters")


## Comparing Linkage Methods

Let's compare different linkage criteria.


In [ ]:
# Compare different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
results = []

for method in linkage_methods:
    clustering = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = clustering.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    results.append({
        'method': method,
        'silhouette': sil_score,
        'labels': labels
    })
    print(f"{method.capitalize()} Linkage: Silhouette Score = {sil_score:.3f}")

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for idx, result in enumerate(results):
    row = idx // 2
    col = idx % 2
    axes[row, col].scatter(X[:, 0], X[:, 1], c=result['labels'], 
                          cmap='viridis', s=50, alpha=0.7)
    axes[row, col].set_title(f"{result['method'].capitalize()} (Silhouette: {result['silhouette']:.3f})")
    axes[row, col].set_xlabel('Feature 1')
    axes[row, col].set_ylabel('Feature 2')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_method = max(results, key=lambda x: x['silhouette'])
print(f"\nBest linkage method: {best_method['method']} (Silhouette: {best_method['silhouette']:.3f})")


## Validation & Testing

Let's validate the clustering and evaluate performance.


In [ ]:
# Evaluate clustering
evaluation = evaluate_clustering(X, y_pred, algorithm='Hierarchical')
print("Clustering Evaluation:")
print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")
print(f"  Number of clusters: {evaluation['n_clusters']}")

# Test different numbers of clusters
k_range = range(2, 8)
silhouette_scores = []

for k in k_range:
    clustering = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = clustering.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    silhouette_scores.append(sil_score)
    print(f"  k={k}: Silhouette Score = {sil_score:.3f}")

optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"\nOptimal number of clusters: {optimal_k} (Silhouette: {max(silhouette_scores):.3f})")

# Plot silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.grid(True, alpha=0.3)
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal k={optimal_k}')
plt.legend()
plt.tight_layout()
plt.show()

# Assertions
assert evaluation['silhouette_score'] > 0, "Silhouette score should be positive"
assert clustering.n_clusters == 4, "Expected 4 clusters"
print("\n✓ Validation checks passed")


## Real-World Application

Let's apply hierarchical clustering to the Iris dataset.


In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Scale features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply Hierarchical Clustering
clustering_iris = AgglomerativeClustering(n_clusters=3, linkage='ward')
y_iris_pred = clustering_iris.fit_predict(X_iris_scaled)

# Evaluate
sil_score_iris = silhouette_score(X_iris_scaled, y_iris_pred)
print("Iris Dataset Clustering:")
print(f"  Number of clusters: {clustering_iris.n_clusters}")
print(f"  Silhouette Score: {sil_score_iris:.3f}")

# Create dendrogram
linkage_matrix_iris = linkage(X_iris_scaled, method='ward')
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix_iris, truncate_mode='level', p=3)
plt.title('Dendrogram for Iris Dataset (Ward Linkage)')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

# Visualize clusters (using first two features)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('True Labels')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Hierarchical Clustering (k=3, Ward)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **Hierarchical Clustering Basics**
   - Builds tree of clusters (dendrogram)
   - Two approaches: agglomerative (bottom-up) and divisive (top-down)
   - No need to specify number of clusters in advance
   - Produces interpretable hierarchy

2. **Linkage Criteria**
   - **Single**: Minimum distance between clusters (chaining effect)
   - **Complete**: Maximum distance (compact clusters)
   - **Average**: Average distance (balanced)
   - **Ward**: Minimizes variance (spherical clusters, works with Euclidean)

3. **Dendrogram**
   - Visual representation of cluster hierarchy
   - Height represents distance at which clusters merge
   - Can extract clusters at any level by cutting dendrogram

4. **Best Practices**
   - Use Ward linkage with Euclidean distance (most common)
   - Scale features before clustering
   - Visualize dendrogram to choose number of clusters
   - Use silhouette score to validate clustering
   - Consider computational cost for large datasets

### When to Use Hierarchical Clustering

✅ **Good for:**
- Unknown number of clusters
- Need to explore cluster structure at multiple levels
- Want interpretable cluster hierarchy
- Small to medium datasets
- Non-spherical clusters
- When dendrogram visualization is helpful

❌ **Not ideal for:**
- Very large datasets (computationally expensive O(n³))
- When you know exact number of clusters (use K-Means)
- Real-time clustering (slow)
- Memory constraints (stores full distance matrix)
- When speed is critical

### Next Steps

- Compare with **K-Means** for known number of clusters
- Try **DBSCAN** for density-based clustering
- Use **PCA** before clustering for high-dimensional data
- Explore **Divisive Clustering** (top-down approach)
- Apply to **gene expression data** for biological clustering
